<a href="https://colab.research.google.com/github/bayuaripermana/bayuaripermana/blob/main/dataset_rumah_sakit_indonesia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()

RAW_FILE = "Hospital_Indonesia_datasets.csv"

df = pd.read_csv(RAW_FILE, sep=";")

# ID adalah identifier, bukan angka untuk dihitung
df["id"] = df["id"].astype(str)

cleaning_log = []
cleaning_log.append({
    "tahap": "Data mentah",
    "jumlah_baris": len(df),
    "jumlah_kolom": df.shape[1]
})

df.head()

Saving Hospital_Indonesia_datasets.csv to Hospital_Indonesia_datasets (3).csv


,id,nama,propinsi,kab,alamat,jenis,kelas,status_blu,kepemilikan,total_tempat_tidur,total_layanan,total_tenaga_kerja
0,1110053,RS Arun Lhokseumawe,Aceh,Kota Lhokseumawe,Jl. Plaju Komplek Perumahan PT Arun Batuphat T...,Rumah Sakit Umum,C,Non BLU/BLUD,SWASTA/LAINNYA,218,36,328
1,1106014,RS Umum Fandika,Aceh,Aceh Tengah,Jl. Terminal Simpang Wariji Blangkolak 1 Kec. ...,Rumah Sakit Umum,D,Non BLU/BLUD,SWASTA/LAINNYA,45,15,45
2,1171110,RS Umum Daerah Meuraxa,Aceh,Kota Banda Aceh,Jl. Soekarno Hatta Km. 2 Desa Mibo Kecamatan B...,Rumah Sakit Umum,B,BLUD,Pemkot,310,77,487
3,1171163,RS Gigi Mulut Universitas Syiah Kuala,Aceh,Kota Banda Aceh,Jl. Prof A. Madjid Ibrahim I No. 5 Banda Aceh ...,Rumah Sakit Khusus Gigi dan Mulut,B,BLU,Kementerian Lain,11,24,0
4,1102027,RS Umum Daerah Kota Subulussalam,Aceh,Kota Subulussalam,Jl. Hamzah Fansyuri (Subulussalam-Rundeng) K...,Rumah Sakit Umum,C,BLUD,Pemkot,189,34,537


In [27]:
def profiling_report(data):
    report = []

    for col in data.columns:
        n_missing = data[col].isna().sum()
        n_empty_string = 0

        if data[col].dtype == "object":
            n_empty_string = (data[col].astype(str).str.strip() == "").sum()

        report.append({
            "kolom": col,
            "missing_nan": n_missing,
            "string_kosong": n_empty_string,
            "jumlah_nilai_unik": data[col].nunique(dropna=False)
        })

    return pd.DataFrame(report)

profile_before = profiling_report(df)
profile_before

,kolom,missing_nan,string_kosong,jumlah_nilai_unik
0,id,0,0,3155
1,nama,0,0,3091
2,propinsi,0,0,38
3,kab,0,0,511
4,alamat,0,0,3155
5,jenis,0,1,21
6,kelas,0,0,8
7,status_blu,0,0,7
8,kepemilikan,0,0,22
9,total_tempat_tidur,0,0,405


In [28]:
print("Duplikat baris penuh:", df.duplicated().sum())
print("Duplikat ID:", df["id"].duplicated().sum())

df.describe()

Duplikat baris penuh: 0
Duplikat ID: 0


,total_tempat_tidur,total_layanan,total_tenaga_kerja
count,3155.000000,3155.000000,3155.000000
mean,135.433914,41.097306,261.980666
std,550.960235,30.583893,382.352731
min,0.000000,1.000000,0.000000
25%,54.000000,22.000000,56.500000
50%,102.000000,31.000000,153.000000
75%,155.000000,50.000000,335.000000
max,30343.000000,419.000000,7939.000000


In [29]:
df_clean = df.copy()

text_cols = df_clean.select_dtypes(include="object").columns

for col in text_cols:
    df_clean[col] = df_clean[col].str.strip().str.replace(r"\s+", " ", regex=True)

# Ubah string kosong menjadi NaN
for col in text_cols:
    df_clean[col] = df_clean[col].replace("", np.nan)

cleaning_log.append({
    "tahap": "Rapikan spasi dan ubah string kosong menjadi NaN",
    "jumlah_baris": len(df_clean),
    "jumlah_kolom": df_clean.shape[1]
})

In [30]:
shifted_ids = ["3101002", "3310015", "3471085", "9228002"]

mask_shifted = df_clean["id"].isin(shifted_ids)

df_clean.loc[mask_shifted, "jenis"] = df_clean.loc[mask_shifted, "kelas"]
df_clean.loc[mask_shifted, "kelas"] = df_clean.loc[mask_shifted, "status_blu"]
df_clean.loc[mask_shifted, "status_blu"] = df_clean.loc[mask_shifted, "kepemilikan"]
df_clean.loc[mask_shifted, "kepemilikan"] = np.nan

# Nilai tempat tidur pada baris ini tidak dapat dipercaya
df_clean.loc[mask_shifted, "total_tempat_tidur"] = np.nan

df_clean.loc[mask_shifted, [
    "id", "nama", "jenis", "kelas", "status_blu",
    "kepemilikan", "total_tempat_tidur"
]]

cleaning_log.append({
    "tahap": "Perbaiki 4 baris dengan pergeseran kolom",
    "jumlah_baris": len(df_clean),
    "jumlah_kolom": df_clean.shape[1]
})

In [31]:
# Propinsi
propinsi_map = {
    "Yogyakarta": "DI Yogyakarta"
}

df_clean["propinsi"] = df_clean["propinsi"].replace(propinsi_map)

# Jenis rumah sakit
valid_jenis = [
    "Rumah Sakit Umum",
    "Rumah Sakit Khusus Ibu dan Anak",
    "Rumah Sakit Khusus Jiwa",
    "Rumah Sakit Khusus Mata",
    "Rumah Sakit Khusus Gigi dan Mulut",
    "Rumah Sakit Khusus Bedah",
    "Rumah Sakit Khusus Jantung",
    "Rumah Sakit Khusus Paru",
    "Rumah Sakit Khusus Orthopedi",
    "Rumah Sakit Khusus Kanker",
    "Rumah Sakit Khusus THT-KL",
    "Rumah Sakit Khusus Infeksi",
    "Rumah Sakit Khusus Ginjal",
    "Rumah Sakit Khusus Stroke",
    "Rumah Sakit Khusus Otak",
    "Rumah Sakit Ketergantungan Obat",
    "RS Kapal/Bergerak"
]

df_clean["jenis"] = df_clean["jenis"].where(
    df_clean["jenis"].isin(valid_jenis),
    "Tidak Valid/Unknown"
)

# Kelas rumah sakit
valid_kelas = ["A", "B", "C", "D", "D PRATAMA"]

df_clean["kelas"] = df_clean["kelas"].where(
    df_clean["kelas"].isin(valid_kelas),
    "Belum Ditetapkan"
)

# Status BLU/BLUD
valid_status = ["BLU", "BLUD", "Non BLU/BLUD"]

df_clean["status_blu"] = df_clean["status_blu"].where(
    df_clean["status_blu"].isin(valid_status),
    "Tidak Valid/Unknown"
)

# Kepemilikan
valid_kepemilikan = [
    "SWASTA/LAINNYA", "Pemkab", "Perusahaan", "Organisasi Sosial",
    "Pemprop", "Pemkot", "Organisasi Islam", "TNI AD", "Perorangan",
    "POLRI", "Organisasi Katholik", "Kementerian Lain", "Kemkes",
    "BUMN", "Organisasi Protestan", "TNI AL", "TNI AU",
    "Organisasi Hindu", "Organisasi Budha"
]

df_clean["kepemilikan"] = df_clean["kepemilikan"].where(
    df_clean["kepemilikan"].isin(valid_kepemilikan),
    "Tidak Valid/Unknown"
)

cleaning_log.append({
    "tahap": "Standardisasi kategori dengan mapping eksplisit",
    "jumlah_baris": len(df_clean),
    "jumlah_kolom": df_clean.shape[1]
})

In [32]:
# Tenaga kerja 0 kemungkinan besar berarti tidak tercatat,
# bukan benar-benar tidak memiliki tenaga kerja.
df_clean["total_tenaga_kerja"] = df_clean["total_tenaga_kerja"].replace(0, np.nan)

df_clean["flag_tenaga_kerja_tidak_tercatat"] = np.where(
    df["total_tenaga_kerja"] == 0,
    "Ya",
    "Tidak"
)

# Tempat tidur 0 tetap dipertahankan sebagai 0,
# karena RS tanpa tempat tidur memang mungkin ada.
df_clean["flag_tempat_tidur_nol"] = np.where(
    df_clean["total_tempat_tidur"] == 0,
    "Ya",
    "Tidak"
)

cleaning_log.append({
    "tahap": "Ubah total_tenaga_kerja = 0 menjadi NaN dan buat flag",
    "jumlah_baris": len(df_clean),
    "jumlah_kolom": df_clean.shape[1]
})

In [33]:
numeric_cols = [
    "total_tempat_tidur",
    "total_layanan",
    "total_tenaga_kerja"
]

outlier_summary = []

for col in numeric_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr

    n_outlier = (df_clean[col] > upper).sum()

    outlier_summary.append({
        "kolom": col,
        "q1": q1,
        "q3": q3,
        "batas_atas_iqr": upper,
        "jumlah_outlier": n_outlier
    })

pd.DataFrame(outlier_summary)

,kolom,q1,q3,batas_atas_iqr,jumlah_outlier
0,total_tempat_tidur,54.0,154.5,305.25,150
1,total_layanan,22.0,50.0,92.00,266
2,total_tenaga_kerja,85.0,372.0,802.50,174


In [34]:
batas_tt = df_clean["total_tempat_tidur"].quantile(0.99)

df_viz = df_clean[
    df_clean["total_tempat_tidur"].notna()
    & (df_clean["total_tempat_tidur"] <= batas_tt)
].copy()

print(f"Batas persentil 99 total_tempat_tidur: {batas_tt}")
print("Data untuk visualisasi:", df_viz.shape)

Batas persentil 99 total_tempat_tidur: 573.0
Data untuk visualisasi: (3119, 14)


In [36]:
# Rasio tenaga kerja per tempat tidur
df_clean["rasio_tenaga_kerja_per_tt"] = np.where(
    (df_clean["total_tempat_tidur"] > 0)
    & df_clean["total_tenaga_kerja"].notna(),
    df_clean["total_tenaga_kerja"] / df_clean["total_tempat_tidur"],
    np.nan
)

# Kategori kapasitas
def kategori_kapasitas(tt):
    if pd.isna(tt):
        return "Tidak Diketahui"
    elif tt == 0:
        return "Tanpa Tempat Tidur"
    elif tt <= 50:
        return "Kecil (1-50 TT)"
    elif tt <= 100:
        return "Menengah (51-100 TT)"
    elif tt <= 250:
        return "Besar (101-250 TT)"
    else:
        return "Sangat Besar (>250 TT)"

df_clean["kategori_kapasitas"] = df_clean["total_tempat_tidur"].apply(kategori_kapasitas)

# Wilayah pulau
pulau_map = {
    "Aceh": "Sumatera", "Sumatera Utara": "Sumatera", "Sumatera Barat": "Sumatera",
    "Riau": "Sumatera", "Kepulauan Riau": "Sumatera", "Jambi": "Sumatera",
    "Sumatera Selatan": "Sumatera", "Bengkulu": "Sumatera", "Lampung": "Sumatera",
    "Kepulauan Bangka Belitung": "Sumatera",
    "DKI Jakarta": "Jawa", "Jawa Barat": "Jawa", "Jawa Tengah": "Jawa",
    "Jawa Timur": "Jawa", "Banten": "Jawa", "DI Yogyakarta": "Jawa",
    "Bali": "Bali & Nusa Tenggara", "Nusa Tenggara Barat": "Bali & Nusa Tenggara",
    "Nusa Tenggara Timur": "Bali & Nusa Tenggara",
    "Kalimantan Barat": "Kalimantan", "Kalimantan Tengah": "Kalimantan",
    "Kalimantan Selatan": "Kalimantan", "Kalimantan Timur": "Kalimantan",
    "Kalimantan Utara": "Kalimantan",
    "Sulawesi Utara": "Sulawesi", "Sulawesi Tengah": "Sulawesi",
    "Sulawesi Selatan": "Sulawesi", "Sulawesi Tenggara": "Sulawesi",
    "Sulawesi Barat": "Sulawesi", "Gorontalo": "Sulawesi",
    "Maluku": "Maluku & Papua", "Maluku Utara": "Maluku & Papua",
    "Papua": "Maluku & Papua", "Papua Barat": "Maluku & Papua",
    "Papua Tengah": "Maluku & Papua", "Papua Selatan": "Maluku & Papua",
    "Papua Pegunungan": "Maluku & Papua", "Papua Barat Daya": "Maluku & Papua"
}

df_clean["wilayah_pulau"] = df_clean["propinsi"].map(pulau_map)

# Kelompok kepemilikan
def kelompok_kepemilikan(x):
    if x in ["Pemkab", "Pemkot", "Pemprop", "Kemkes", "Kementerian Lain"]:
        return "Pemerintah"
    elif x in ["TNI AD", "TNI AL", "TNI AU", "POLRI"]:
        return "TNI/Polri"
    elif x in ["Perusahaan", "BUMN", "Perorangan", "SWASTA/LAINNYA"]:
        return "Swasta/Lainnya"
    elif x.startswith("Organisasi"):
        return "Organisasi Keagamaan/Sosial"
    else:
        return "Tidak Valid/Unknown"

df_clean["kelompok_kepemilikan"] = df_clean["kepemilikan"].apply(kelompok_kepemilikan)

# Kota atau Kabupaten
df_clean["tipe_daerah"] = np.where(
    df_clean["kab"].str.startswith("Kota", na=False),
    "Kota",
    "Kabupaten"
)

cleaning_log.append({
    "tahap": "Tambah fitur rasio, kapasitas, pulau, kepemilikan, dan tipe daerah",
    "jumlah_baris": len(df_clean),
    "jumlah_kolom": df_clean.shape[1]
})

In [37]:
assert df_clean["id"].is_unique, "Terdapat ID duplikat"
assert df_clean["id"].dtype == object, "ID harus bertipe string"

assert df_clean["kelas"].isin(
    ["A", "B", "C", "D", "D PRATAMA", "Belum Ditetapkan"]
).all(), "Terdapat kelas tidak valid"

assert df_clean["status_blu"].isin(
    ["BLU", "BLUD", "Non BLU/BLUD", "Tidak Valid/Unknown"]
).all(), "Terdapat status BLU/BLUD tidak valid"

assert df_clean["wilayah_pulau"].notna().all(), "Terdapat provinsi belum terpetakan"

print("Semua validasi cleaning lulus.")

Semua validasi cleaning lulus.


In [38]:
log_df = pd.DataFrame(cleaning_log)
log_df

,tahap,jumlah_baris,jumlah_kolom
0,Data mentah,3155,12
1,Rapikan spasi dan ubah string kosong menjadi NaN,3155,12
2,Perbaiki 4 baris dengan pergeseran kolom,3155,12
3,Standardisasi kategori dengan mapping eksplisit,3155,12
4,Ubah total_tenaga_kerja = 0 menjadi NaN dan bu...,3155,14
5,"Tambah fitur rasio, kapasitas, pulau, kepemili...",3155,19


In [39]:
# Simpan dua versi:
# 1) Data bersih lengkap
# 2) Data siap visualisasi

df_clean.to_csv(
    "hospital_indonesia_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

df_viz.to_csv(
    "hospital_indonesia_for_visualization.csv",
    index=False,
    encoding="utf-8-sig"
)

files.download("hospital_indonesia_clean.csv")
files.download("hospital_indonesia_for_visualization.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>